# lu_astra — Ultimate 8ФМ Classic Policy Training

**Goal**: Train the strongest possible 8ФМ Classic RL policy targeting 60+ average score.

**Architecture**: v3 features (~320 state + ~80 action), cross-attention fusion, 4 residual blocks, 512 hidden dim.

**Training**: 8000 PPO updates, 1024 games/update, curriculum learning, 12-hour budget.

**Hardware**: Kaggle GPU (T4×2 or P100), 12-hour session limit.

In [ ]:
# Install dependencies
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install -q wandb

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

In [ ]:
# Clone repository
import os
import subprocess

REPO_URL = "https://github.com/leonka123/8FM_AI.git"
REPO_DIR = "/kaggle/working/8FM_AI"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

# Add to path
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# Training Configuration
CONFIG = {
    # Training scale
    "updates": 8000,
    "games_per_update": 1024,
    "batch_size": 2048,
    "ppo_epochs": 12,
    
    # Learning rate schedule
    "learning_rate": 5e-4,
    "learning_rate_final": 1e-5,
    "warmup_steps": 500,
    
    # PPO hyperparameters
    "entropy_coef": 0.05,
    "entropy_coef_final": 0.001,
    "clip_eps": 0.2,
    "clip_eps_final": 0.1,
    "gae_lambda": 0.97,
    "gamma": 0.995,
    "target_kl": 0.02,
    "max_grad_norm": 0.5,
    "value_clip": 0.5,
    "huber_delta": 1.0,
    
    # Evaluation
    "eval_every": 40,
    "eval_games": 100,
    
    # Checkpoints
    "snapshot_every": 20,
    "snapshot_pool_size": 20,
    
    # Curriculum
    "player_counts": "3,4,5,6,7",
    "seed": 42,
    
    # Output
    "output": "models/lu_astra.json",
    "resume": None,  # Set to "models/lu_astra-latest.json" to resume
    
    # Logging
    "wandb": True,
    "wandb_project": "8fm-astra",
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Run training
import subprocess
import sys

cmd = [
    sys.executable, "kaggle/train_lu_astra.py",
    f"--updates={CONFIG['updates']}",
    f"--games-per-update={CONFIG['games_per_update']}",
    f"--batch-size={CONFIG['batch_size']}",
    f"--ppo-epochs={CONFIG['ppo_epochs']}",
    f"--learning-rate={CONFIG['learning_rate']}",
    f"--learning-rate-final={CONFIG['learning_rate_final']}",
    f"--warmup-steps={CONFIG['warmup_steps']}",
    f"--entropy-coef={CONFIG['entropy_coef']}",
    f"--entropy-coef-final={CONFIG['entropy_coef_final']}",
    f"--clip-eps={CONFIG['clip_eps']}",
    f"--clip-eps-final={CONFIG['clip_eps_final']}",
    f"--gae-lambda={CONFIG['gae_lambda']}",
    f"--gamma={CONFIG['gamma']}",
    f"--target-kl={CONFIG['target_kl']}",
    f"--max-grad-norm={CONFIG['max_grad_norm']}",
    f"--value-clip={CONFIG['value_clip']}",
    f"--huber-delta={CONFIG['huber_delta']}",
    f"--eval-every={CONFIG['eval_every']}",
    f"--eval-games={CONFIG['eval_games']}",
    f"--snapshot-every={CONFIG['snapshot_every']}",
    f"--snapshot-pool-size={CONFIG['snapshot_pool_size']}",
    f"--player-counts={CONFIG['player_counts']}",
    f"--seed={CONFIG['seed']}",
    f"--device=cuda",
    f"--output={CONFIG['output']}",
]

if CONFIG["resume"]:
    cmd.append(f"--resume={CONFIG['resume']}")

if CONFIG["wandb"]:
    cmd.append("--wandb")
    cmd.append(f"--wandb-project={CONFIG['wandb_project']}")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
print(f"Exit code: {result.returncode}")

In [ ]:
# Quick evaluation of trained model
import torch
from eightfm.torch_astra import AstraNetwork, state_features_v3, BatchedClassicEnvV3, astra_to_rl_network
from eightfm.rl import save_rl, load_rl
from eightfm.baselines import ComboPolicy, GreedyPolicy, SupervisedPolicy
from eightfm.rl import RLGreedyPolicy
from eightfm.policy import load_policy
from eightfm.rl_train import evaluate_controlled_seat
from eightfm.torch_astra import AstraPolicy
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
rl_net = load_rl("models/lu_astra.json")
print(f"Loaded model: feature_version={rl_net.feature_version}")

# Create AstraNetwork
test_env = BatchedClassicEnvV3(player_count=4, batch_size=1, device=device, seed=0)
test_obs = test_env.observation()
test_state = state_features_v3(test_obs, seat=0)
state_dim = test_state.shape[1]
action_dim = 80

model = AstraNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=512,
    num_residual_blocks=4,
    num_heads=8,
    feature_version=3,
    seed=42,
).to(device)

# Note: weights don't map exactly due to architecture difference
# For true evaluation, use the saved JSON with NumPy RLNetwork

# Evaluate using NumPy interface
from eightfm.rl import RLGreedyPolicy
numpy_policy = RLGreedyPolicy(rl_net)

opponents = {
    "combo": ComboPolicy(),
    "greedy": GreedyPolicy(),
    "supervised": SupervisedPolicy(load_policy("models/classic-policy.json")),
    "rl_classic": RLGreedyPolicy(load_rl("models/classic-rl.json")),
    "lu_luna": RLGreedyPolicy(load_rl("models/lu_luna.json")),
    "lu_sol": RLGreedyPolicy(load_rl("models/lu_sol.json")),
}

print("\nEvaluation (20 games/opponent/player_count):")
for name, opp in opponents.items():
    win_rates = []
    scores = []
    for n in (3, 4, 5, 6, 7):
        report = evaluate_controlled_seat(numpy_policy, opp, player_count=n, games=20, seed=10000+n*100)
        win_rates.append(report["win_rate"])
        scores.append(report["mean_score"])
    print(f"  vs {name:12s}: win={np.mean(win_rates):.3f}, score={np.mean(scores):.2f}")

In [ ]:
# Run tournament with lu_astra included
!python -X utf8 -m eightfm tournament \
    --participants luna,sol,astra,supervised,greedy,rl_classic \
    --players 3,4,5,6,7 \
    --games 100 \
    --astra models/lu_astra.json \
    --output tournament_astra_report.json \
    --seed-base 120000 \
    --seed 0

In [ ]:
# Download artifacts
from google.colab import files
import os

artifacts = [
    "models/lu_astra.json",
    "models/lu_astra-latest.json",
    "models/lu_astra-metrics.json",
    "tournament_astra_report.json",
]

for f in artifacts:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)
    else:
        print(f"Not found: {f}")